# Acesso e verificação — Parquet SRAG

Setup, download, checagem de metadados, leitura com filtro e smoke test do pré-processador esparso.

In [ ]:
from pathlib import Path
import os
import sys
import time

# ⚠️ Antes de qualquer import numérico: limitar OpenBLAS/OpenMP/MKL
for _var in ("OPENBLAS_NUM_THREADS", "OMP_NUM_THREADS",
             "MKL_NUM_THREADS", "NUMEXPR_NUM_THREADS"):
    os.environ.setdefault(_var, "1")

import numpy as np
import pandas as pd
import pyarrow.parquet as pq
from IPython.display import display

ROOT_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(ROOT_DIR) not in sys.path:
    sys.path.insert(0, str(ROOT_DIR))

from src.utils import env
from src.utils import config as C
from src.utils.storage import get_parquet, read_srag

env.carregar_dotenv()
env.garantir_diretorios()
print("ROOT_DIR:", ROOT_DIR)
print(env.resumo())

ROOT_DIR: c:\Users\ivanI\OneDrive\Documentos\Projetos\Projeto_Integrador
ROOT_DIR          = C:\Users\ivanI\OneDrive\Documentos\Projetos\Projeto_Integrador
ENV               = local
.env encontrado?  = False
GITHUB_TOKEN?     = False


In [2]:
print(f"Repo: {C.GITHUB_REPO} | Versão: {C.DATA_VERSION} | Arquivo: {C.PARQUET_NAME}")
print(f"URL: {C.srag_parquet_url()}")

parquet_path = get_parquet()
print(f"Parquet: {parquet_path}")
print(f"Tamanho: {parquet_path.stat().st_size / 1024**2:.2f} MB")

Repo: Ivan-S-Oliveira/Projeto_Integrador | Versão: dados-v2 | Arquivo: srag.parquet
URL: https://github.com/Ivan-S-Oliveira/Projeto_Integrador/releases/download/dados-v2/srag.parquet
Parquet: C:\Users\ivanI\OneDrive\Documentos\Projetos\Projeto_Integrador\data\processed\dados-v2\srag.parquet
Tamanho: 289.51 MB


In [3]:
inicio = time.perf_counter()
arquivo = pq.ParquetFile(parquet_path)
meta = arquivo.metadata

if meta.num_rows <= 0 or meta.num_columns <= 0:
    raise ValueError("Parquet vazio ou sem colunas.")

lote = next(arquivo.iter_batches(batch_size=5), None)
if lote is None:
    raise ValueError("Não foi possível extrair amostra.")

amostra = lote.to_pandas(types_mapper=pd.ArrowDtype).reset_index(drop=True)
display(amostra)
display(pq.read_schema(parquet_path))
print(
    f"registros={meta.num_rows:,} | colunas={meta.num_columns} | "
    f"row_groups={meta.num_row_groups} | tempo={time.perf_counter() - inicio:.3f}s"
)

,id_registro,ano_arquivo,data_snapshot,arquivo_origem,nu_notific,dt_notific,sem_not,dt_sin_pri,sem_pri,sg_uf_not,...,vg_oms,vg_omsout,vg_lin,vg_met,vg_metout,vg_dtres,vg_enc,vg_reinf,vg_codest,reinf
0,0,2026,2026-09-28,INFLUD26-28-09-2026.parquet,31781201987679,2026-06-11,24,2026-06-02,23,BA,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2
1,1,2026,2026-09-28,INFLUD26-28-09-2026.parquet,31775652598724,2026-03-23,13,2026-03-22,13,MA,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2
2,2,2026,2026-09-28,INFLUD26-28-09-2026.parquet,31781205405890,2026-03-29,14,2026-03-23,13,MT,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2
3,3,2026,2026-09-28,INFLUD26-28-09-2026.parquet,31782407823707,2026-06-17,25,2026-06-15,25,SP,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2
4,4,2026,2026-09-28,INFLUD26-28-09-2026.parquet,31773493207022,2026-03-14,11,2026-03-09,11,SE,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2


id_registro: int64
ano_arquivo: int16
data_snapshot: date32[day]
arquivo_origem: string
nu_notific: string
dt_notific: string
sem_not: string
dt_sin_pri: string
sem_pri: string
sg_uf_not: string
id_regiona: string
co_regiona: string
id_municip: string
co_mun_not: string
cs_sexo: string
dt_nasc: string
nu_idade_n: string
tp_idade: string
cod_idade: string
cs_gestant: string
cs_raca: string
cs_etinia: string
cs_escol_n: string
id_pais: string
co_pais: string
sg_uf: string
id_rg_resi: string
co_rg_resi: string
id_mn_resi: string
co_mun_res: string
cs_zona: string
nosocomial: string
ave_suino: string
febre: string
tosse: string
garganta: string
dispneia: string
desc_resp: string
saturacao: string
diarreia: string
vomito: string
outro_sin: string
outro_des: string
fator_risc: string
puerpera: string
cardiopati: string
hematologi: string
sind_down: string
hepatica: string
asma: string
diabetes: string
neurologic: string
pneumopati: string
imunodepre: string
renal: string
obesidade: string
ob

registros=4,666,910 | colunas=198 | row_groups=28 | tempo=0.075s


In [ ]:
df = read_srag(columns=["sg_uf"])
print("shape:", df.shape)
display(df.head())

df_sp = read_srag(
    columns=["dt_notific", "sg_uf", "evolucao"],
    filters=[("sg_uf", "==", "SP")],
)
print("SP shape:", df_sp.shape)
display(df_sp.head())

shape: (4666910, 1)


,sg_uf
0,BA
1,MA
2,MT
3,SP
4,SE


SP shape: (1301367, 3)


,dt_notific,sg_uf,evolucao
0,2026-06-17,SP,NaN
1,2026-05-13,SP,1
2,2026-08-25,SP,2
3,2026-06-20,SP,1
4,2026-01-19,SP,1


: 

In [ ]:
from scipy import sparse
from threadpoolctl import threadpool_limits
from src.models.pipeline import (
    FEATURES_CATEGORICAS,
    FEATURES_NUMERICAS,
    build_preprocessor,
)

threadpool_limits(limits=1)   # cinto + suspensório, vale para o resto da sessão

rng = np.random.default_rng(42)
N = 10_000

dados = {}
for i, col in enumerate(FEATURES_NUMERICAS):
    v = rng.normal(size=N).astype(np.float32)
    if i == 0:
        v[::100] = np.nan
    dados[col] = v

for col in FEATURES_CATEGORICAS:
    v = rng.choice(["1", "2", "9"], size=N, p=[0.4375, 0.4375, 0.125]).astype(object)
    v[rng.random(N) < 0.20] = None
    dados[col] = pd.Series(v, dtype="category")

X = pd.DataFrame(dados)
prep = build_preprocessor(
    numericas=FEATURES_NUMERICAS,
    categoricas=FEATURES_CATEGORICAS,
    saida="esparsa",
)
M = prep.fit_transform(X)

assert sparse.issparse(M), f"Esperado esparso, veio {type(M).__name__}"
assert M.dtype == np.dtype("float32"), f"dtype {M.dtype}"
assert M.shape[0] == N

nnz = sparse.csr_matrix(M).count_nonzero()
dens = nnz / (M.shape[0] * M.shape[1])
print(f"shape={M.shape} | nnz={nnz:,} | densidade={dens:.2%} | dtype={M.dtype}")
print("OK: matriz esparsa float32.")